In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os 
path = r"C:\Users\Admin\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1"
print(os.listdir(path))

['WA_Fn-UseC_-Telco-Customer-Churn.csv']


In [4]:
df = pd.read_csv(path + r"\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [30]:
#Exploration
# df.info()
df.head()
# df.shape
# df.describe()
# print(df.isnull().sum())
# print(f"Churn distribution:\n{df['Churn'].value_counts(normalize=True)}")


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
# Data Preprocessing
df1 = df.copy()

In [6]:
df1['TotalCharges'] = pd.to_numeric(df1['TotalCharges'], errors='coerce')
df1.dropna(inplace=True)
df1.drop('customerID', axis=1, inplace=True)
df1['Churn'] = df1['Churn'].map({"No": 0, "Yes": 1})

In [7]:
# Feature engineering -- Create a feature with reserching and domain knowledge
df1['IsMonthToMonthContract'] = (df1['Contract'] == 'Month-to-month').astype(int) 
df1['IsElectronicCheck'] = (df1['PaymentMethod'] == 'Electronic check').astype(int)
df1['NoInternetService'] = (df1['InternetService'] == 'No').astype(int)

In [29]:
df1['TenureGroup'] = pd.cut(df1['tenure'], bins=[0, 6, 12, 24, 48, 72], 
                            labels=['0-6m', '6-12m', '12-24m', '24-48m', '48m+'])
df1 = pd.get_dummies(df1, columns=['TenureGroup'], drop_first=False)

In [9]:
# uses intensity ratio
df1['AvgMOnthlySpend'] = df1['MonthlyCharges'] / (df1['TotalCharges'] + 1)

In [10]:
# How many services does each customer actually use
service_cols = ['PhoneService', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df1['NumServices'] = (df1[service_cols] != 'No').sum(axis=1)

In [11]:
# one hot encoding remaining categorical variables
df1 = pd.get_dummies(df1, drop_first=True)
bool_cols = df1.select_dtypes(include='bool').columns
df1[bool_cols] = df1[bool_cols].astype(int)

In [12]:
# Declaring X and y
X = df1.drop('Churn',axis=1)
y = df1['Churn']
X

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,IsMonthToMonthContract,IsElectronicCheck,NoInternetService,TenureGroup_0-6m,TenureGroup_6-12m,TenureGroup_12-24m,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,1,1,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,1,1,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,24,84.80,1990.50,0,0,0,0,0,1,...,0,1,0,1,1,0,1,0,0,1
7039,0,72,103.20,7362.90,0,0,0,0,0,0,...,0,1,0,1,1,0,1,1,0,0
7040,0,11,29.60,346.45,1,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
7041,1,4,74.40,306.60,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1


In [17]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42, stratify=y)

In [18]:
# Handle class imbalance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42,k_neighbors=5)
X_train_smote,y_train_smote = smote.fit_resample(X_train,y_train)

In [19]:
X_train_smote.shape
y_train_smote.value_counts()

Churn
0    4130
1    4130
Name: count, dtype: int64

In [20]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_trained_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

In [21]:
from xgboost import XGBClassifier
# Hyperparameter Tuning
xgb_param_grid = {
    'max_depth':[3,4,5],
    'learning_rate':[0.01,0.03,0.05],
    'n_estimators':[300,400,500],
    'subsample':[0.7,0.8],
    'colsample_bytree':[0.7,0.8],
}

xgb_search = GridSearchCV(
    XGBClassifier(
        scale_pos_weight=(y_train_smote.value_counts()[0] / y_train_smote.value_counts()[1]),
        eval_metric='logloss',
        random_state=42,
        verbosity=0
    ),
    xgb_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
xgb_search.fit(X_trained_scaled, y_train_smote)
print(f"Best XGBoost params: {xgb_search.best_params_}")
print(f"Best cross-val ROC-AUC: {xgb_search.best_score_:.4f}")

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best XGBoost params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 500, 'subsample': 0.7}
Best cross-val ROC-AUC: 0.9305


In [22]:
# random forest hyperparametergrid
from sklearn.ensemble import RandomForestClassifier
rf_param_grid = {
    'max_depth':[8,10,12],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'n_estimators': [100, 200, 300],
}
rf_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
rf_search.fit(X_trained_scaled, y_train_smote)
print(f"Best RF params: {rf_search.best_params_}")
print(f"Best cross-val ROC-AUC: {rf_search.best_score_:.4f}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best RF params: {'max_depth': 12, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 300}
Best cross-val ROC-AUC: 0.9226


In [23]:
from sklearn.metrics import classification_report, roc_auc_score

best_xgb = xgb_search.best_estimator_
best_rf = rf_search.best_estimator_

print("XGBoost:\n", classification_report(y_test, best_xgb.predict(X_test_scaled)))
print("Random Forest:\n", classification_report(y_test, best_rf.predict(X_test_scaled)))

XGBoost:
               precision    recall  f1-score   support

           0       0.85      0.83      0.84      1033
           1       0.57      0.61      0.59       374

    accuracy                           0.77      1407
   macro avg       0.71      0.72      0.71      1407
weighted avg       0.78      0.77      0.77      1407

Random Forest:
               precision    recall  f1-score   support

           0       0.87      0.80      0.84      1033
           1       0.55      0.68      0.61       374

    accuracy                           0.77      1407
   macro avg       0.71      0.74      0.72      1407
weighted avg       0.79      0.77      0.78      1407



In [25]:
import joblib
joblib.dump(best_xgb, "models/model_xgb_v1.pkl")
joblib.dump(best_rf, "models/model_rf_v1.pkl")
joblib.dump(scaler, "models/scaler_v1.pkl")
joblib.dump(X.columns.tolist(), "models/feature_columns_v1.pkl")

['models/feature_columns_v1.pkl']

In [ ]:
import sys
sys.path.append(".")
from app.preprocessing import engineer_features

raw_example = {
    "tenure": 24,
    "Contract": "Month-to-month",
    "MonthlyCharges": 89.5
}

result = engineer_features(raw_example)
print(result)

In [27]:
import joblib

cols = joblib.load("../models/feature_columns_v1.pkl")
print(cols)

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'IsMonthToMonthContract', 'IsElectronicCheck', 'NoInternetService', 'TenureGroup_0-6m', 'TenureGroup_6-12m', 'TenureGroup_12-24m', 'TenureGroup_24-48m', 'TenureGroup_48m+', 'AvgMOnthlySpend', 'NumServices', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']
